In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class FakeCIFAR10(Dataset):
    def __init__(self, num_samples=1000, image_size=224, num_classes=10):
        self.num_samples = num_samples
        self.image_size = image_size
        self.num_classes = num_classes

        # تصاویر رندوم: [N, 3, 224, 224]
        self.images = torch.randn(num_samples, 3, image_size, image_size)
        # لیبل رندوم بین 0 تا num_classes-1
        self.labels = torch.randint(0, num_classes, (num_samples,))

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

# train / val ساده
train_dataset = FakeCIFAR10(num_samples=800, image_size=224, num_classes=10)
val_dataset   = FakeCIFAR10(num_samples=200, image_size=224, num_classes=10)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=3, patch_size=16, embed_dim=256):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv2d(
            in_channels=in_channels,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        # x: [B, 3, 224, 224]
        x = self.proj(x)          # [B, embed_dim, 14, 14] اگر 224/16=14
        x = x.flatten(2)          # [B, embed_dim, 196]
        x = x.transpose(1, 2)     # [B, 196, embed_dim]
        return x

class ViTEmbedding(nn.Module):
    def __init__(self, image_size=224, patch_size=16, embed_dim=256):
        super().__init__()
        self.patch_embed = PatchEmbedding(
            in_channels=3,
            patch_size=patch_size,
            embed_dim=embed_dim
        )
        num_patches = (image_size // patch_size) ** 2  # 14*14=196

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim))

    def forward(self, x):
        x = self.patch_embed(x)          # [B, 196, embed_dim]
        B = x.shape[0]

        cls_tokens = self.cls_token.expand(B, -1, -1)   # [B, 1, embed_dim]
        x = torch.cat([cls_tokens, x], dim=1)           # [B, 197, embed_dim]
        x = x + self.pos_embed                          # [B, 197, embed_dim]
        return x

class SimpleViT(nn.Module):
    def __init__(
        self,
        image_size=224,
        patch_size=16,
        num_classes=10,
        embed_dim=256,
        depth=6,
        num_heads=8,
        mlp_dim=512,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = ViTEmbedding(
            image_size=image_size,
            patch_size=patch_size,
            embed_dim=embed_dim
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=mlp_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=depth
        )

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)        # [B, 197, embed_dim]
        x = self.encoder(x)          # [B, 197, embed_dim]
        cls_output = x[:, 0]         # [B, embed_dim]
        cls_output = self.norm(cls_output)
        logits = self.head(cls_output)   # [B, num_classes]
        return logits


In [ ]:
model = SimpleViT(
    image_size=224,
    patch_size=16,
    num_classes=10,
    embed_dim=256,
    depth=6,
    num_heads=8,
    mlp_dim=512,
    dropout=0.1
).to(device)


for param in model.embedding.parameters():
    param.requires_grad = False

for param in model.encoder.parameters():
    param.requires_grad = False  


for param in model.head.parameters():
    param.requires_grad = True   

for param in model.norm.parameters():
    param.requires_grad = True     


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

In [ ]:
def claculate_accuracy(logits, labels):
    preds = torch.argmax(logits, dim=1)
    correct = (preds == labels).sum().item()
    total = labels.size(0)
    return correct / total

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    running_acc = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)

        loss = criterion(logits,labels)

        loss.backward()

        optimizer.step()

        acc =  calculate_accuracy(logits, labels)
        
        running_loss += loss.item()
        running_acc += acc

    epoch_loss = running_loss / len(loader)
    epoch_acc = running_acc / len(loader)  

    return epoch_loss, epoch_acc

In [ ]:
def validate_one_epoch(model, loader, criterion, device):
    model.eval()  # حالت eval: Dropout خاموش، BatchNorm/LayerNorm در حالت eval

    running_loss = 0.0
    running_acc = 0.0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            acc = calculate_accuracy(logits, labels)

            running_loss += loss.item()
            running_acc += acc

    epoch_loss = running_loss / len(loader)
    epoch_acc = running_acc / len(loader)

    return epoch_loss, epoch_acc


In [ ]:
num_epochs = 10
best_val_loss = float("inf")
best_state_dict = None

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, device
    )

    val_loss, val_acc = validate_one_epoch(
        model, val_loader, criterion, device
    )

    scheduler.step()
    print(f"[Epoch {epoch:02d}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state_dict = model.state_dict()


if best_state_dict is not None:
    model.load_state_dict(best_state_dict)
    print("Loaded best model based on validation loss.")        